# Hybrid RAG Evaluation: neural-bridge/rag-dataset-1200

We test the rag system by asking the LLM used in the app questions without context, after which we save the context as texts in our database and prompt it again, but include the queries the second time. We use another judge model for comparison, and we will also compare them by hand.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "datasets", "requests"], check=True)

CompletedProcess(args=['/Users/sofijatasevska/DevArchive/NLP-SMIM PROEKT/unpaid_intern/.venv/bin/python', '-m', 'pip', 'install', 'datasets', 'requests'], returncode=0)

In [2]:
import requests
import json
import io
import os
import time
from datasets import load_dataset

API_URL     = "http://127.0.0.1:8000"
USER_ID     = "eval_user_001"
SAMPLE_SIZE = 50 
TOP_K       = 3
RESULTS_DIR = "eval_results"

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Config ready.")

Config ready.


/Users/sofijatasevska/DevArchive/NLP-SMIM PROEKT/unpaid_intern/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load dataset

In [4]:
raw     = load_dataset("neural-bridge/rag-dataset-1200", split="train")
dataset = list(raw.select(range(SAMPLE_SIZE)))

print(f"Loaded {len(dataset)} samples")
print(json.dumps(dataset[0], indent=2))

Loaded 50 samples
{
  "context": "Francisco Rogers found the answer to a search query collar george herbert essay\nLink ----> collar george herbert essay\nWrite my essay ESSAYERUDITE.COM\nconstitution research paper ideas\ndefinition essay humility\nbusiness strategy case study solution\ncorporals course essay\ndecisions in paradise essays\ncollege essay word count\ncredit cart terminal paper\nbyron don juan essay\ndemocratic party essays\ncoursework language learning material teaching\nchristmas commercialized essay\ndahrendorf essays theory society\nbuy apa format essay buy apa format essay\nconan doyle speckled band essay\ncollege essay application prompt\ncolumbia university mfa creative writing acceptance rate\ncrucible coursework questions\ncollege essay topics texas\ncover letter thesis proposal\nciting ma thesis\ncompare and contrast essays for elementary\ncoursework completed without degree\ncomparison islam christianity essay\ncheerleading stereotypes essay\ncultural diversit

## 2. Helpers

In [14]:
def safe_request(method, url, **kwargs):
    try:
        r = requests.request(method, url, timeout=120, **kwargs)
        if not r.ok:
            print(f"[HTTP {r.status_code}] {url} — {r.text[:200]}")
            return None
        return r.json()
    except Exception as e:
        print(f"[ERROR] {url}: {e}")
        return None

def call_no_context(question: str) -> str:
    """POST query, no retrieval context."""
    result = safe_request(
        "post",
        f"{API_URL}/query",
        json={"query": edit_prompt(question=question), "max_tokens": 300, "temperature": 0.1}
    )
    return result.get("response", "") if result else "ERROR: no response"


def query_vector(question: str, top_k: int = TOP_K) -> list:
    result = safe_request(
        "get",
        f"{API_URL}/query/vector",
        params={"user_id": USER_ID, "query": edit_prompt(question=question), "top_k": top_k}
    )
    return result.get("results", []) if result else []


def query_graph(question: str) -> list:
    result = safe_request(
        "get",
        f"{API_URL}/query/graph",
        params={"user_id": USER_ID, "query": edit_prompt(question=question)}
    )
    return result.get("results", []) if result else []


def call_rag(question: str, use_vector=True, use_graph=True) -> dict:
    """POST /rag/query — retrieval-augmented answer"""
    vector_results = query_vector(question) if use_vector else []
    graph_results  = query_graph(question)  if use_graph  else []

    result = safe_request(
        "post",
        f"{API_URL}/rag/query",
        json={
            "query":          edit_prompt(question=question),
            "vector_results": vector_results,
            "graph_results":  graph_results,
            "max_tokens":     300,
            "temperature":    0.1
        }
    )
    return result or {}


JUDGE_PROMPT = """You are an answer evaluator.
Given a question, a ground truth answer, and a model answer,
respond with ONLY a JSON object, nothing else:
{{"score": <0 or 1>, "reason": "<one sentence>"}}
Score 1 if the model answer is factually correct and addresses the question.
Score 0 if it is wrong, hallucinated, or says it does not know.

Question: {question}
Ground truth: {ground_truth}
Model answer: {model_answer}"""

def judge_answer(question: str, ground_truth: str, model_answer: str) -> dict:
    
    prompt = JUDGE_PROMPT.format(
        question=question,
        ground_truth=ground_truth,
        model_answer=model_answer
    )
    raw = call_no_context(prompt)
    try:
        clean = raw.strip().strip("```json").strip("```").strip()
        return json.loads(clean)
    except Exception as e:
        return {"score": 0, "reason": f"Judge parse failed: {e} | raw: {raw[:100]}"}

def save_results(filename: str, results: list):
    path = os.path.join(RESULTS_DIR, filename)
    with open(path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Saved {len(results)} results → {path}")

def score_summary(results: list) -> dict:
    scores = [r["score"] for r in results if "score" in r]
    if not scores:
        return {"total": 0, "correct": 0, "accuracy": 0.0}
    return {
        "total":    len(scores),
        "correct":  sum(scores),
        "accuracy": round(sum(scores) / len(scores), 3)
        
    }
    
    
def edit_prompt(question) -> str:
    return f"Answer the question without additional sentences. Question: {question}"

In [15]:
health = safe_request("get", f"{API_URL}/health")
if not health:
    raise RuntimeError("Server not responding.")
print("Server healthy:", health)

Server healthy: {'status': 'healthy', 'vector_storage': True, 'graph_storage': True}


## 2. Part 1 (no context)
Calls `POST /query`, no documents, no retrieval, same model our RAG uses.

In [16]:
baseline_results = []

for i, row in enumerate(dataset):
    question     = row["question"]
    ground_truth = row["answer"]

    print(f"[{i+1}/{len(dataset)}] {question[:120]}")

    model_answer = call_no_context(question)
    judgment     = judge_answer(question, ground_truth, model_answer)

    baseline_results.append({
        "index":        i,
        "question":     question,
        "ground_truth": ground_truth,
        "model_answer": model_answer,
        "score":        judgment["score"],
        "reason":       judgment["reason"]
    })
    
    print(f"model answer: {model_answer}")

    time.sleep(0.5)

save_results("baseline.json", baseline_results)
print("\nBaseline summary:", score_summary(baseline_results))

[1/50] Who found the answer to a search query collar george herbert essay?
model answer: The answer to the search query "collar george herbert essay" was found by a user or researcher conducting the search.
[2/50] What are some of the potential negative impacts of charity as discussed in the context?
model answer: Some potential negative impacts of charity include creating dependency, undermining local economies, fostering a culture of entitlement, and potentially perpetuating systemic issues rather than addressing root causes.
[3/50] Who were the three stars in the NHL game between Buffalo Sabres and Edmonton Oilers?
model answer: The three stars in the NHL game between the Buffalo Sabres and Edmonton Oilers were Connor McDavid, Leon Draisaitl, and Jeff Skinner.
[4/50] What services does Pearl Moving Company in Santa Clarita, 91390 offer?
model answer: Pearl Moving Company in Santa Clarita, 91390 offers residential and commercial moving services, packing and unpacking, loading and unl

## 4. Part 2 (uploading documents)
Each dataset row has a `context` field, we upload it as a plain text document.

In [20]:
def upload_document(user_id: str, doc_name: str, content: str):
    r = requests.post(
        f"{API_URL}/upload",
        data={"user_id": user_id, "document_name": doc_name},
        files={"file": (doc_name, io.BytesIO(content.encode("utf-8")), "text/plain")},
        timeout=60
    )
    if r.status_code == 200:
        return r.json()
    print(f"Upload failed '{doc_name}': {r.status_code} {r.text[:150]}")
    return None


upload_errors = 0
for i, row in enumerate(dataset):
    doc_name = f"doc_{i:04d}.txt"
    context  = row["context"]
    content  = " ".join(context) if isinstance(context, list) else context
    result = upload_document(USER_ID, doc_name, content)

    if result:
        statuses = [r.get("status") for r in result]
        if all(s == "skipped" for s in statuses):
            status = "skipped (duplicate)"
        elif any(s == "error" for s in statuses):
            status = f"partial error — {[r.get('message','') for r in result if r.get('status') == 'error']}"
        else:
            status = "uploaded"
    else:
        status = "FAILED"
        upload_errors += 1

    if i % 10 == 0 or status == "FAILED":
        print(f"[{i+1}/{len(dataset)}] {doc_name} — {status}")

print(f"\nErrors: {upload_errors}/{len(dataset)}")

[1/50] doc_0000.txt — partial error — ['1 validation error for DocumentUploadResponse\ndocument_id\n  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]\n    For further information visit https://errors.pydantic.dev/2.7/v/string_type']
[11/50] doc_0010.txt — uploaded
[21/50] doc_0020.txt — uploaded
[31/50] doc_0030.txt — uploaded
[41/50] doc_0040.txt — uploaded

Errors: 0/50


## 5. Evaluation runner
Shared function for `POST /rag/query`.

In [23]:
def run_evaluation(mode: str, use_vector: bool, use_graph: bool) -> list:
    print(f"  {mode.upper()}  (vector={use_vector}, graph={use_graph})")

    results = []

    for i, row in enumerate(dataset):
        question     = row["question"]
        ground_truth = row["answer"]

        print(f"[{i+1}/{len(dataset)}] {question[:120]}")

        rag_result   = call_rag(question, use_vector=use_vector, use_graph=use_graph)
        model_answer = rag_result.get("response", "ERROR: no response")
        context_used = rag_result.get("context_used", {})
        
        print(f"model answer: {model_answer}")

        judgment = judge_answer(question, ground_truth, model_answer)

        results.append({
            "index":        i,
            "mode":         mode,
            "question":     question,
            "ground_truth": ground_truth,
            "model_answer": model_answer,
            "context_used": context_used,
            "score":        judgment["score"],
            "reason":       judgment["reason"]
        })

        time.sleep(0.5)

    save_results(f"{mode}.json", results)
    print(f"\n{mode} summary:", score_summary(results))
    return results

## 6. Vector-only

In [24]:
vector_results = run_evaluation("vector_only", use_vector=True, use_graph=False)

  VECTOR_ONLY  (vector=True, graph=False)
[1/50] Who found the answer to a search query collar george herbert essay?
model answer: Francisco Rogers found the answer to a search query collar george herbert essay.
[2/50] What are some of the potential negative impacts of charity as discussed in the context?
model answer: Charity can sometimes make things worse by not addressing underlying systemic issues.
[3/50] Who were the three stars in the NHL game between Buffalo Sabres and Edmonton Oilers?
model answer: I don't have enough information to answer that.
[4/50] What services does Pearl Moving Company in Santa Clarita, 91390 offer?
model answer: Pearl Moving Company in Santa Clarita, 91390 offers a variety of moving services including apartment movers, local movers, moving company, moving services, office move, home movers, out of state movers, professional movers, moving companies, cheap movers, movers reviews, movers cost, home moving companies, moving and storage companies, long dist

These are already amazing results! Without context, the model's precision was only about 0.38, even less so if the documents uploaded on our server are subjective.

## 7. Graph-only

In [25]:
graph_results = run_evaluation("graph_only", use_vector=False, use_graph=True)

  GRAPH_ONLY  (vector=False, graph=True)
[1/50] Who found the answer to a search query collar george herbert essay?
model answer: I don't have enough information to answer that.
[2/50] What are some of the potential negative impacts of charity as discussed in the context?
model answer: Some potential negative impacts of charity include:

1. Creating dependency and undermining self-reliance
2. Perpetuating poverty by not addressing root causes
3. Fostering a sense of superiority in donors and inferiority in recipients
4. Being used as a tool for political or religious influence
5. Diverting attention from systemic issues and structural inequalities
6. Potentially being inefficient or mismanaged
7. Creating a cycle of aid that is difficult to break
[3/50] Who were the three stars in the NHL game between Buffalo Sabres and Edmonton Oilers?
model answer: I don't have enough information to answer that.
[4/50] What services does Pearl Moving Company in Santa Clarita, 91390 offer?
model answe

The model answered most of the questions with "I don't have enough informacion.". 

## 8. Hybrid

In [ ]:
hybrid_results = run_evaluation("hybrid", use_vector=True, use_graph=True)

  HYBRID  (vector=True, graph=True)
[1/50] Who found the answer to a search query collar george herbert essay?
model answer: Francisco Rogers found the answer to a search query collar george herbert essay.
[2/50] What are some of the potential negative impacts of charity as discussed in the context?
model answer: Some potential negative impacts of charity include: it can be a bandaid placed on broken systems, it may not correct underlying systemic issues, and sometimes it can make things worse.
[3/50] Who were the three stars in the NHL game between Buffalo Sabres and Edmonton Oilers?
model answer: I don't have enough information to answer that.
[4/50] What services does Pearl Moving Company in Santa Clarita, 91390 offer?
model answer: Pearl Moving Company in Santa Clarita, 91390 offers a variety of moving services, including apartment movers, local movers, office moves, home movers, out of state movers, professional movers, moving companies, cheap movers, movers reviews, movers cost, 

## 9. Comparison summary

In [ ]:
all_runs = {
    "baseline":    baseline_results,
    "vector_only": vector_results,
    "graph_only":  graph_results,
    "hybrid":      hybrid_results,
}

print(f"{'Mode':<15} {'Total':>7} {'Correct':>9} {'Accuracy':>10}")
for mode, res in all_runs.items():
    s = score_summary(res)
    print(f"{mode:<15} {s['total']:>7} {s['correct']:>9} {s['accuracy']:>10.1%}")

In [ ]:
print("Per-question delta: hybrid vs baseline\n")
print(f"{'#':<5} {'Base':>5} {'Hybrid':>7} {'Delta':>7}  Question")

for b, h in zip(baseline_results, hybrid_results):
    delta  = h["score"] - b["score"]
    marker = "CORRECT" if delta > 0 else ("incorrect" if delta < 0 else "")
    print(f"{b['index']:<5} {b['score']:>5} {h['score']:>7} {delta:>+7}{marker}  {b['question'][:60]}")

In [ ]:
comparison = [
    {
        "index":        i,
        "question":     baseline_results[i]["question"],
        "ground_truth": baseline_results[i]["ground_truth"],
        "baseline":     {"answer": baseline_results[i]["model_answer"], "score": baseline_results[i]["score"]},
        "vector_only":  {"answer": vector_results[i]["model_answer"],   "score": vector_results[i]["score"]},
        "graph_only":   {"answer": graph_results[i]["model_answer"],    "score": graph_results[i]["score"]},
        "hybrid":       {"answer": hybrid_results[i]["model_answer"],   "score": hybrid_results[i]["score"]},
    }
    for i in range(len(dataset))
]

save_results("full_comparison.json", comparison)
print("saved to eval_results/")